In [0]:
data = [("B1", "DrugA", "Plant1"), 
        ("B2", "DrugA", "Plant2"), 
        ("B3", "DrugB", "Plant1")]
schema = ["batch_id", "product", "facility"]
df_batches = spark.createDataFrame(data, schema)
df_batches.show(truncate=False) 

data = [("T1", "B1", "purity","pass"),
         ("T2", "B1", "stability", "fail"), 
         ("T3", "B2", "purity", "pass")]
schema = ["test_id", "batch_id", "test_type", "result"]
df_tests = spark.createDataFrame(data, schema) 

data = [("D1", "B1", "temperature excursion")]
schema = ["deviation_id", "batch_id", "description"]
df_deviations = spark.createDataFrame(data, schema) 

# q1 show all batches and their quality test results: batch_id, test_type, result 
# ANSWER
df_q1 = df_tests.select("batch_id", "test_type", "result") 

# q2 show all batches and their quality test results, including batches with no tests: batch_id, test_type, result 
df_bt = df_batches.select("batch_id")
df_q2 = df_bt.join(df_tests, on="batch_id", how="left")
df_bt2 = df_q2.select("batch_id", "test_type", "result")
df_bt2.show(truncate=False)

# q3 find batches with both failed tests and deviations: batch_id 
df_q3 = df_q2.filter(df_q2.result == "fail").join(df_deviations, on="batch_id", how="inner").select("batch_id").distinct()
df_q3.show(truncate=False)

# q4 show batch-level counts of tests and deviations: batch_id, test_count, deviation_count 
from pyspark.sql.functions import count
df_q4 = df_q2.groupBy("batch_id").agg(count("test_type").alias("test_count")).join(df_deviations.groupBy("batch_id").agg(count("deviation_id").alias("deviation_count")), on="batch_id", how="inner")
df_q4.show(truncate=False)

# q5 find batches who no deviations: batch_id 
df_q5 = df_batches.join(df_deviations, on="batch_id", how="left").filter(df_deviations.deviation_id.isNull()).select("batch_id")
df_q5.show(truncate=False)